# 2.7c — SVM SOTA : LIBSVM sous le capot de `sklearn.svm.SVC`

**Navigation** : [<< 2.7b-SMO-From-Scratch](2.7b-SMO-From-Scratch.ipynb) | [Index](README.md) | [2.8-Théorie-PAC >>](2.8-Theorie-PAC.ipynb)

**Kernel** : Python 3 · **Durée estimée** : ~30 min

## Introduction

> **Concept-phare** : [2.7b](2.7b-SMO-From-Scratch.ipynb) a écrit la boucle SMO à la main et l'a validée contre `sklearn.svm.SVC`. Ce notebook inverse la perspective : le solveur industriel devient le sujet. `SVC` est l'enveloppe Python de **LIBSVM** (Chang & Lin, 2011) — du C++ compilé, un working set au second ordre (Fan, Chen & Lin, 2005), du shrinking et un cache de noyau. On mesure ce que ce packaging achète : exactitude, temps, vecteurs supports — puis on ouvre **le dual que SVC ne rend pas** (`dual_coef_`), on chiffre le **gap de dualité et les violations KKT côté LIBSVM** avec la même machinerie que 2.7b, et on actionne les molettes que la boîte noire expose (`tol`).

**Position dans le parcours** : le **pendant SOTA** de [2.7b](2.7b-SMO-From-Scratch.ipynb). Ici, on examine le solveur industriel ; [2.11b — Proximal Operators](2.11b-Proximal-Operators-From-Scratch.ipynb) traite séparément le Lasso avec ISTA/FISTA, `sklearn.linear_model` et `cvxpy`.

### Objectifs d'apprentissage

1. **Mesurer** le geste SOTA honnêtement : protocole de temps répété (un fit LIBSVM sur 140 points dure moins qu'une milliseconde — un chrono naïf affiche 0.00), exactitude et vecteurs supports sur le **même dataset que 2.7b**.
2. **Reconstruire le dual** que `SVC` ne retourne pas : `alpha` complet depuis `dual_coef_` + `support_`, et vérifier la contrainte d'égalité `Σ α_i y_i = 0`.
3. **Évaluer la solution côté LIBSVM** : objectif dual, objectif primal reconstruit, gap de dualité, violations KKT résiduelles — les mêmes instruments que 2.7b, pointés sur l'autre solveur.
4. **Actionner les molettes de production** : sweep de `tol`, et en exercice, `GridSearchCV` sur `C × gamma`.

### Prérequis

- [2.7b — SMO from scratch](2.7b-SMO-From-Scratch.ipynb) : le dual soft-margin, la condition KKT, le gap de dualité (ce notebook réutilise ces instruments sans les redériver).
- [2.7 — Modèles non paramétriques](2.7-Modeles-Non-Parametriques.ipynb) : SVM, kernel trick, vecteurs supports.

NumPy et scikit-learn suffisent ; le noyau RBF vient de `sklearn.metrics.pairwise` (le côté SOTA jusqu'au bout).

> **Référence.** Chang, C.-C. & Lin, C.-J. (2011), *LIBSVM: a library for support vector machines*, ACM TIST 2(27). Fan, R.-E., Chen, P.-H. & Lin, C.-J. (2005), *Working set selection using second order information for training support vector machines*, JMLR 6.

[Jumeau C#](2.7c-SVM-SOTA-Comparison-CSharp.ipynb)

### Vérification de l'environnement

In [1]:
import time

import numpy as np
import sklearn
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score, pairwise
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

print(f"numpy={np.__version__}  scikit-learn={sklearn.__version__}")

numpy=2.3.5  scikit-learn=1.8.0


### Lecture du résultat : l'environnement d'exécution

Tout tourne sur CPU en quelques secondes. La comparaison avec 2.7b n'a de sens que sur le même dataset, la même graine et les mêmes hyperparamètres — c'est la section suivante.

## 1. Le même dataset que 2.7b, à la graine près

Le protocole de 2.7b section 7 est recopié **exactement** : `make_moons(200, noise=0.25, random_state=42)`, labels en {-1, +1}, split stratifié 70/30 (`random_state=42`), `C = 1.0`, `gamma = 2.0`, `tol = 1e-3`. La seule variable expérimentale est le solveur : LIBSVM ici, la SMO de Platt maison dans 2.7b.

In [2]:
X, y_pm = make_moons(n_samples=200, noise=0.25, random_state=42)
y = np.where(y_pm == 0, -1, 1).astype(float)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

C, GAMMA, TOL = 1.0, 2.0, 1e-3

print(f"train : {len(y_tr)} points   test : {len(y_te)} points   (2.7b : identique)")
print(f"C={C}  gamma={GAMMA}  tol={TOL:.0e}   (2.7b : identique)")

train : 140 points   test : 60 points   (2.7b : identique)
C=1.0  gamma=2.0  tol=1e-03   (2.7b : identique)


### Lecture du résultat : le terrain est posé

Mêmes points, même split, mêmes hyperparamètres qu'en 2.7b : à la fin du notebook, toute différence de résultat ne pourra être attribuée qu'au solveur.

## 2. Mesurer les ajustements à froid

Un ajustement SVC sur ce petit jeu est rapide et un chronométrage unique peut être bruité. On mesure une moyenne de 50 ajustements, avec un nouvel estimateur à chaque répétition. Les durées sont propres à la machine et au protocole ; pour une comparaison avec 2.7b, consulter sa sortie courante et tenir compte des opérations incluses.

In [3]:
# Mesurer des ajustements froids sur le même découpage, sans réutiliser le modèle.
clf = SVC(C=C, kernel="rbf", gamma=GAMMA, tol=TOL, max_iter=20000)
clf.fit(X_tr, y_tr)

acc_skl = accuracy_score(y_te, clf.predict(X_te))
n_sv = int(clf.n_support_.sum())

N_REPETS = 50
t0 = time.perf_counter()
for _ in range(N_REPETS):
    SVC(C=C, kernel="rbf", gamma=GAMMA, tol=TOL, max_iter=20000).fit(X_tr, y_tr)
t_skl = (time.perf_counter() - t0) / N_REPETS

print(f"sklearn SVC (LIBSVM) : accuracy test = {acc_skl:.3f}, |SV| = {n_sv}, "
      f"temps moyen = {t_skl * 1e3:.2f} ms  (moyenne de {N_REPETS} fits a froid)")
print("Pour les mesures SMO et leur durée, consulter la sortie de 2.7b.")

sklearn SVC (LIBSVM) : accuracy test = 0.967, |SV| = 48, temps moyen = 0.83 ms  (moyenne de 50 fits a froid)
Pour les mesures SMO et leur durée, consulter la sortie de 2.7b.


### Lire les mesures

La sortie donne accuracy et nombre de supports sur ce tirage ; les valeurs de 2.7b sont rappelées dans le tableau, à ses tolérances. Un accord de prédictions et de cardinalité ne prouve pas une égalité exacte des coefficients. La moyenne de 50 fits réduit le bruit d'un chronométrage unique, sans isoler à elle seule les causes de performance. `SVC.n_iter_` et les examens de la boucle SMO ne comptent pas les mêmes opérations.

Sur cette instance, les sorties rapportent 0,967 d'accuracy et 48 supports pour LIBSVM, comme dans le relevé Python de 2.7b. Celui-ci compare aussi les ensembles de supports et les scores de décision : ces mesures complètent le seul décompte et restent liées à ce tirage et aux tolérances. Le temps affiché est celui des ajustements LIBSVM froids ; la durée de 2.7b doit être lue dans son exécution courante. L'implémentation compilée et le choix du working set contribuent à des coûts différents, mais ce protocole ne quantifie pas séparément leurs effets. Le compteur `n_iter_` mesure les itérations internes du solveur et ne se convertit pas directement en examens SMO.


## Exercices

Trois exercices progressifs : le premier fait varier `C` sur le solveur SOTA et relie le budget de marge au nombre de supports ; le deuxième vérifie en profondeur la reconstruction du dual de la section 3 ; le troisième est le geste de production complet — la recherche d'hyperparamètres par validation croisée.

### Exercice 1 — Faire varier C

Mesurez supports et accuracy pour cinq valeurs de C ; ne supposez pas de monotonie universelle.

In [4]:
# Exercice 1 : l'effet de C sur le nombre de vecteurs supports, cote LIBSVM
# TODO etudiant : completez sweep_C(valeurs) qui, pour chaque C de valeurs,
# entraine un SVC(C=C, kernel='rbf', gamma=2.0, tol=1e-3) sur (X_tr, y_tr),
# et retourne la liste des couples (n_supports, accuracy_test). Faites varier
# C dans [0.01, 0.1, 1, 10, 100] et comparez les nombres de supports et les accuracies observes. Le nombre de
# supports ne doit pas etre suppose monotone pour tout jeu de donnees.


def sweep_C(valeurs):
    # Indice : clf.n_support_.sum() pour le compte ; accuracy_score(y_te, ...).
    # Etape 1 : boucle sur valeurs, fit a froid.
    # Etape 2 : collecter (n_supports, accuracy) de chaque fit.
    result = None  # TODO etudiant
    return result


fixtures=[[.01,.1,1.,10.,100.],[.2,2.,20.]]
answers=[sweep_C(values) for values in fixtures]
if all(value is None for value in answers):
    print("Exercice 1 à compléter.")
elif any(value is None for value in answers):
    print("Exercice 1 : à revoir")
else:
    ok=True
    for valeurs,resultat in zip(fixtures,answers):
        references=[]
        for c_value in valeurs:
            fitted=SVC(C=c_value,kernel='rbf',gamma=2.,tol=1e-3).fit(X_tr,y_tr)
            references.append((int(fitted.n_support_.sum()),accuracy_score(y_te,fitted.predict(X_te))))
        ok=ok and len(resultat)==len(references) and all(a[0]==b[0] and abs(a[1]-b[1])<1e-10 for a,b in zip(resultat,references))
    print("Exercice 1 : correct" if ok else "Exercice 1 : à revoir")

Exercice 1 à compléter.


## 3. Reconstruire le dual que `SVC` ne rend pas

`SVC` n'expose pas le vecteur `alpha` complet — il rend `dual_coef_` (les `y_i · α_i` **des seuls supports**, concaténés) et `support_` (leurs indices dans le train). Le dual complet se reconstruit : `alpha` nul hors supports, `|dual_coef_|` sur les supports. Avec lui, tous les instruments de 2.7b section 10 se pointent sur LIBSVM :

- `||w||²` reconstruit depuis le dual : `Σ_ij (y_i α_i)(y_j α_j) K(x_i, x_j)` ;
- objectif dual `D(α) = Σ α_i − ½ ||w||²` ;
- objectif primal `P = ½ ||w||² + C Σ ξ_i`, les `ξ` lus sur `decision_function` ;
- **gap de dualité** `P − D` et **violations KKT** résiduelles.

In [5]:
# --- reconstruction du dual complet --------------------------------------
alpha = np.zeros(len(y_tr))
alpha[clf.support_] = np.abs(clf.dual_coef_[0])
print(f"alpha reconstruit : {int((alpha > 1e-12).sum())} non nuls "
      f"(n_support_ = {n_sv})   somme = {alpha.sum():.4f}")
print(f"contrainte d'egalite sum(alpha_i * y_i) = {float((alpha * y_tr).sum()):.2e}")

# --- objectifs primal et dual sur la MEME solution -------------------------
ay = clf.dual_coef_[0]
K_ss = pairwise.rbf_kernel(clf.support_vectors_, clf.support_vectors_, gamma=GAMMA)
w2 = float(ay @ K_ss @ ay)
D_libsvm = float(alpha.sum() - 0.5 * w2)

f_tr = clf.decision_function(X_tr)
xi = np.maximum(0.0, 1.0 - y_tr * f_tr)
P_libsvm = float(0.5 * w2 + C * xi.sum())
gap = P_libsvm - D_libsvm

# --- violations KKT residuelles (meme forme que 2.7b section 10) -----------
viol_low = np.where(alpha < C - 1e-12, np.maximum(0.0, 1.0 - y_tr * f_tr), 0.0)
viol_high = np.where(alpha > 1e-12, np.maximum(0.0, y_tr * f_tr - 1.0), 0.0)
V = float(viol_low.sum() + viol_high.sum())

print(f"||w||^2 depuis le dual            = {w2:.4f}")
print(f"objectif dual   D(alpha)          = {D_libsvm:.4f}")
print(f"objectif primal P(w, b, xi)       = {P_libsvm:.4f}")
print(f"gap de dualite  P - D             = {gap:.2e}   (>= 0 : dualite faible)")
print(f"violation KKT residuelle  sum V_i = {V:.2e}")
print(f"  pire violation ponctuelle       = {max(viol_low.max(), viol_high.max()):.2e}")
print(f"repartition des alpha : =0 -> {(alpha <= 1e-6).sum()},  "
      f"0<alpha<C -> {((alpha > 1e-6) & (alpha < C - 1e-6)).sum()},  "
      f"=C -> {(alpha >= C - 1e-6).sum()}")
print("2.7b SMO (commite)  : D = 30.3961, gap = 3.0e-03, sum V = 6.38e-03")

alpha reconstruit : 48 non nuls (n_support_ = 48)   somme = 40.1958
contrainte d'egalite sum(alpha_i * y_i) = 2.22e-16
||w||^2 depuis le dual            = 19.5995
objectif dual   D(alpha)          = 30.3961
objectif primal P(w, b, xi)       = 30.3981
gap de dualite  P - D             = 1.99e-03   (>= 0 : dualite faible)
violation KKT residuelle  sum V_i = 4.12e-03
  pire violation ponctuelle       = 5.22e-04
repartition des alpha : =0 -> 92,  0<alpha<C -> 16,  =C -> 32
2.7b SMO (commite)  : D = 30.3961, gap = 3.0e-03, sum V = 6.38e-03


### Lecture du résultat : LIBSVM s'arrête aussi à `tol` près

- **La contrainte d'égalité est satisfaite numériquement** (`Σ α_i y_i ≈ 10⁻¹⁶`) : la reconstruction depuis `dual_coef_` est fidèle, et c'est l'exercice 2 qui la fait vérifier à fond.
- **Le gap de dualité côté LIBSVM est plus serré** que celui de la SMO maison (2.7b : `3.0 × 10⁻³`) — même ordre de grandeur, epsilon plus petit. Aucun des deux solveurs n'atteint l'optimum exact : **les deux s'arrêtent à `tol = 10⁻³` près**, chacun avec son critère. La complémentarité KKT (points hors marge à `α = 0`, points à `α = C` dans la marge ou au-delà, points sur la marge à `0 < α < C`) se lit sur la répartition des `alpha` comme en 2.7b.
- **Les valeurs duales sont proches aux tolérances mesurées** : 2.7b avait déjà évalué le `D(α)` de LIBSVM en contre-épreuve (écart `4 × 10⁻⁵` avec sa propre solution). Ici on le mesure depuis l'intérieur : c'est la même mathématique, conditionnée par un arrêt approché assumé.

## 4. Le tableau comparatif : from scratch (2.7b) vs SOTA (LIBSVM)

Les colonnes 2.7b citent les valeurs **committées** de son exécution (même machine, même dataset) ; les colonnes LIBSVM sont mesurées ci-dessus. Les lignes de code comptent le code spécifique au solveur : `rbf_kernel` + sous-problème 2D + cascade de Platt + `smo_fit` pour 2.7b, contre l'appel `SVC(...).fit(...)` ici.

In [6]:
# Les valeurs SMO historiques sont séparées du temps LIBSVM mesuré ici.
SMO_D, SMO_GAP, SMO_ACC, SMO_SV, SMO_ITER, SMO_LOC = 30.3961, 3.0e-3, 0.967, 48, 13020, 150

print("=" * 96)
print(f"{'solveur':<26}{'D(alpha)':>10}{'gap P-D':>10}{'accuracy':>10}"
      f"{'|SV|':>6}{'temps':>10}{'n_iter':>9}{'LOC':>6}")
print("-" * 96)
print(f"{'LIBSVM via SVC (mesure)':<26}{D_libsvm:>10.4f}{gap:>10.1e}{acc_skl:>10.3f}"
      f"{n_sv:>6}{t_skl * 1e3:>8.2f}ms{'n/d':>9}{2:>6}")
print(f"{'SMO Platt maison (2.7b)':<26}{SMO_D:>10.4f}{SMO_GAP:>10.1e}{SMO_ACC:>10.3f}"
      f"{SMO_SV:>6}{'voir 2.7b':>10}{SMO_ITER:>9}{SMO_LOC:>6}")
print("=" * 96)
print("n_iter LIBSVM : accessible via SVC.n_iter_, non releve ici ; compteur distinct de la boucle SMO maison.")

solveur                     D(alpha)   gap P-D  accuracy  |SV|     temps   n_iter   LOC
------------------------------------------------------------------------------------------------
LIBSVM via SVC (mesure)      30.3961   2.0e-03     0.967    48    0.83ms      n/d     2
SMO Platt maison (2.7b)      30.3961   3.0e-03     0.967    48 voir 2.7b    13020   150
n_iter LIBSVM : accessible via SVC.n_iter_, non releve ici ; compteur distinct de la boucle SMO maison.


### Pourquoi comparer les deux approches ?

Sur ce tirage, les valeurs duales, les supports et les prédictions sont proches aux tolérances affichées. La boucle from scratch rend visibles les paires, le biais et les résidus ; SVC fournit le solveur LIBSVM et son interface de production. Les temps doivent être lus dans chaque environnement et les compteurs n'ont pas la même unité. Les diagnostics de dual et KKT complètent l'accuracy sans transformer un arrêt approché en preuve d'identité exacte.

Le code de 2.7b expose la mise à jour d'une paire de multiplicateurs, la reconstruction du biais et la vérification des conditions KKT. L'interface SVC fournit `predict`, `decision_function` et l'intégration dans les pipelines et la validation croisée. Ces deux perspectives répondent à des besoins complémentaires : comprendre les mécanismes aide à diagnostiquer un arrêt anticipé ou un avertissement de convergence ; les bibliothèques facilitent l'emploi sur d'autres données. Les comparaisons numériques restent dépendantes des critères et tolérances indiqués.


### Exercice 2 : certifier la reconstruction du dual

La section 3 reconstruit `alpha` depuis `dual_coef_` et constate `Σ α_i y_i ≈ 10⁻¹⁶`. Une vérification plus forte compare les **scores de décision** recalculés depuis le dual reconstruit à ceux de `clf.decision_function`.

In [7]:
# Exercice 2 : le dual reconstruit reproduit-il decision_function ?
# TODO etudiant : ecrivez ecart_reconstruction(clf, X) qui recalcule le score
# de decision f(x) = sum_i alpha_i y_i K(x_i, x) + b sur CHAQUE ligne de X
# depuis (alpha reconstruit, y_tr, X_tr, clf.intercept_), en utilisant
# pairwise.rbf_kernel(X, X_tr, gamma), et retourne l'ecart max
# |f_recalcule - clf.decision_function(X)|. Resultat attendu : ~1e-15.


def ecart_reconstruction(clf, X):
    # Indice : les alpha non supports valent 0 -- la somme se reduit aux
    # supports, mais LAISSER la somme complete sur X_tr est plus simple et
    # numeriquement equivalent. Le biais est clf.intercept_[0].
    # Etape 1 : reconstruire alpha (section 3).
    # Etape 2 : f = (alpha * y_tr) @ rbf_kernel(X, X_tr).T + b.
    # Etape 3 : ecart max avec clf.decision_function(X).
    result = None  # TODO etudiant
    return result


class _ScoreDecale:
    def __init__(self, base):
        self.base = base
    def __getattr__(self, name):
        return getattr(self.base, name)
    def decision_function(self, X):
        return self.base.decision_function(X) + .25

resultat = ecart_reconstruction(clf, X_te)
decale = ecart_reconstruction(_ScoreDecale(clf), X_te)
if resultat is None and decale is None:
    print("Exercice 2 à compléter.")
elif resultat is None or decale is None:
    print("Exercice 2 : à revoir")
else:
    ok = np.isfinite(resultat) and np.isfinite(decale) and abs(resultat)<1e-9 and abs(decale-.25)<1e-9
    print("Exercice 2 : correct" if ok else "Exercice 2 : à revoir")

Exercice 2 à compléter.


## 5. Tolérance d'arrêt

On répète le protocole pour quatre tolérances, avec un plafond de 200 000 itérations et 50 fits à froid par valeur. Le tableau calcule à chaque fois accuracy, supports, dual et gap. Resserrer la tolérance modifie le critère d'arrêt ; l'évolution des valeurs et des temps est mesurée pour cette instance, sans garantir une monotonie pour chaque exécution.

In [8]:
print(f"{'tol':>8}{'temps moyen':>14}{'accuracy':>10}{'|SV|':>6}{'D(alpha)':>10}{'gap P-D':>10}")
for tol_i in (1e-2, 1e-3, 1e-4, 1e-5):
    clf_i = SVC(C=C, kernel="rbf", gamma=GAMMA, tol=tol_i, max_iter=200000)
    t0 = time.perf_counter()
    for _ in range(N_REPETS):
        SVC(C=C, kernel="rbf", gamma=GAMMA, tol=tol_i, max_iter=200000).fit(X_tr, y_tr)
    t_i = (time.perf_counter() - t0) / N_REPETS
    clf_i.fit(X_tr, y_tr)
    acc_i = accuracy_score(y_te, clf_i.predict(X_te))

    a_i = np.zeros(len(y_tr))
    a_i[clf_i.support_] = np.abs(clf_i.dual_coef_[0])
    ay_i = clf_i.dual_coef_[0]
    K_i = pairwise.rbf_kernel(clf_i.support_vectors_, clf_i.support_vectors_, gamma=GAMMA)
    w2_i = float(ay_i @ K_i @ ay_i)
    D_i = float(a_i.sum() - 0.5 * w2_i)
    f_i = clf_i.decision_function(X_tr)
    P_i = float(0.5 * w2_i + C * np.maximum(0.0, 1.0 - y_tr * f_i).sum())

    print(f"{tol_i:>8.0e}{t_i * 1e3:>11.2f}ms{acc_i:>10.3f}"
          f"{int(clf_i.n_support_.sum()):>6}{D_i:>10.4f}{P_i - D_i:>10.1e}")

     tol   temps moyen  accuracy  |SV|  D(alpha)   gap P-D


   1e-02       0.80ms     0.967    49   30.3953   1.6e-02


   1e-03       0.89ms     0.967    48   30.3961   2.0e-03


   1e-04       0.97ms     0.967    48   30.3961   1.7e-04


   1e-05       2.57ms     0.967    48   30.3961   2.4e-05


### Lecture du balayage

Sur les sorties de ce tirage, le gap diminue quand on resserre les tolérances de la grille, tandis que l'accuracy et le support se stabilisent. Cela montre le compromis numérique ici, pas que le temps ou le nombre de supports suivent une loi monotone universelle. Une décision sur tol doit tenir compte du besoin de précision, du budget et des données.

Le tableau permet de lire conjointement le dual, le gap, l'accuracy, le nombre de supports et les temps mesurés pour chacune des quatre tolérances. Un gap plus faible donne ici une indication de précision d'optimisation, mais une accuracy inchangée ne démontre pas que les scores ou les multiplicateurs sont identiques. La tolérance utile dépend de l'usage : étudier les supports proches de la borne peut demander une vérification plus fine que la seule prédiction. Les mesures sur ce petit jeu ne prédisent pas le coût sur un autre jeu.


## 6. Ce que LIBSVM fait que notre SMO ne fait pas

Trois mécanismes séparent LIBSVM de la boucle simplifiée inspirée de Platt en 2.7b — aucun ne change l'optimum, tous changent le chemin :

1. **Working set au second ordre** (Fan, Chen & Lin, 2005) : au lieu de balayer les exemples à la recherche d'une violation KKT (cascade de Platt), LIBSVM choisit d'abord un indice admissible selon le gradient, puis un second à l'aide d'une estimation du gain tenant compte de la courbure (les `η`). Cela ne revient pas à maximiser globalement sur toutes les paires ; aucun gain de temps ou d'itérations universel n'est établi ici. L'exercice 3 de 2.7b propose une heuristique simplifiée, sans prétendre reproduire cette sélection.
2. **Shrinking** : certaines variables aux bornes sont provisoirement retirées selon leur statut et les bornes de gradient/KKT ; l'ensemble actif est réexaminé, notamment lors de l'étape de réactivation.
3. **Cache de noyau** : la matrice de Gram ne tient pas en mémoire sur les gros datasets ; LIBSVM cache les colonnes récemment utilisées et recompute le reste. Notre implémentation matérialise `K` en entier : son stockage en `O(n²)` est limité par la mémoire disponible.

La contrepartie du packaging : trajectoire des `α` invisible, paramètres internes (`cache_size`, `shrinking`) réduits à des molettes opaques.

### Exercice 3 : le geste de production — `GridSearchCV` sur `C × gamma`

Le SOTA ne s'arrête pas au solveur : l'écosystème enchaîne la validation croisée sur la grille d'hyperparamètres, avec le scoring de votre choix.

Renvoyez aussi les neuf moyennes de validation pour contrôler toute la grille.

La fonction reçoit les jeux d’entraînement et de test ; elle est contrôlée sur deux jeux distincts.

In [9]:
# Exercice 3 : grille C×gamma par validation croisée sur les données fournies.
def meilleure_grille(X_train,y_train,X_test,y_test):
    # TODO : GridSearchCV SVC RBF, C=[.1,1,10], gamma=[.5,2,8], cv=5, accuracy.
    # Retourner (best_params, best_score_cv, accuracy_test, neuf scores CV).
    return None

# Deux jeux empêchent de confondre un résultat mémorisé avec la procédure.
tiny_X=np.array([[(-1. if i%2==0 else 1.)+.03*i,np.sin(.5*i)] for i in range(40)])
tiny_y=np.array([-1. if i%2==0 else 1. for i in range(40)])
fixtures=[(X_tr,y_tr,X_te,y_te),(tiny_X,tiny_y,tiny_X[:10],tiny_y[:10])]
answers=[meilleure_grille(*fixture) for fixture in fixtures]
if all(result is None for result in answers):
    print("Exercice 3 à compléter.")
elif any(result is None or not isinstance(result,tuple) or len(result)!=4 or any(value is None for value in result) for result in answers):
    print("Exercice 3 : à revoir")
else:
    from sklearn.model_selection import GridSearchCV
    ok=True
    for (X_train,y_train,X_test,y_test),result in zip(fixtures,answers):
        reference=GridSearchCV(SVC(kernel='rbf'),{'C':[.1,1.,10.],'gamma':[.5,2.,8.]},cv=5,scoring='accuracy').fit(X_train,y_train)
        params,score_cv,accuracy,scores=result
        valid=(params==reference.best_params_ and abs(score_cv-reference.best_score_)<1e-10
               and abs(accuracy-accuracy_score(y_test,reference.predict(X_test)))<1e-10
               and np.shape(scores)==(9,) and np.isfinite(scores).all() and np.allclose(scores,reference.cv_results_['mean_test_score']))
        ok=ok and valid
    print("Exercice 3 : correct" if ok else "Exercice 3 : à revoir")

Exercice 3 à compléter.


## Conclusion

Ce notebook a montré que **la boîte noire s'ouvre** : le dual complet se reconstruit depuis `dual_coef_`, le gap de dualité et les violations KKT se mesurent sur LIBSVM exactement comme sur la SMO maison, et la tolérance d'arrêt est un contrat explicite — pas une promesse d'optimalité exacte. Les résultats des deux solveurs sont comparés à leurs tolérances ; un accord empirique ne définit pas à lui seul un problème bien posé.

**Pour aller plus loin** : [2.7b](2.7b-SMO-From-Scratch.ipynb) pour la boucle SMO complète et ses diagnostics visuels ; [2.7](2.7-Modeles-Non-Parametriques.ipynb) pour le cadre SVM/kernel ; [2.11b](2.11b-Proximal-Operators-From-Scratch.ipynb) pour l'autre grande famille de l'optimisation convexe non différentiable.

## Références et provenance

**Repères de production.** Dans l'Epic optimisation convexe ([#16061](https://github.com/jsboige/CoursIA/issues/16061)), ce notebook correspond au bloc B.5 (SVM-SOTA). Le bloc B.6 (Lasso-SOTA et validation croisée de λ) est traité par [2.11c — Lasso](2.11c-Lasso-SOTA-Comparison.ipynb), et le bloc B.7 (CVXPY déclaratif) par [2.11e — CVXPY](2.11e-CVXPY-Optimisation.ipynb). Selon le carnet et l’environnement, les solveurs disponibles incluent CLARABEL et SCS. Les blocs A.1 (SMO) et A.3 (proximal) sont couverts par [2.7b](2.7b-SMO-From-Scratch.ipynb) et 2.11b ; le bloc A.2 est traité dans [2.11d — ADMM](2.11d-Optimisation-ADMM-From-Scratch.ipynb).


1. Chang, C.-C. & Lin, C.-J. (2011). *LIBSVM: a library for support vector machines*. ACM Transactions on Intelligent Systems and Technology, 2(27).
2. Fan, R.-E., Chen, P.-H. & Lin, C.-J. (2005). *Working set selection using second order information for training support vector machines*. Journal of Machine Learning Research, 6.
3. Platt, J. C. (1998). *Sequential Minimal Optimization: A Fast Algorithm for Training Support Vector Machines*. MSR-TR-98-14.
4. Pedregosa, F. et al. (2011). *Scikit-learn: Machine Learning in Python*. JMLR 12.
5. Deisenroth, M.P., Faisal, A.A. & Ong, C.S. (2020). *Mathematics for
Machine Learning*. Cambridge UP. — ch. 12 (Support Vector Machines)
pose la comparaison entre formulations SVM (linéaire vs noyau,
C-SVC vs nu-SVC, hyperparamètres C et gamma) que ce carnet
exploite pour la sélection de modèle SOTA. Le critère
d'équivalence formulation primale-duale est la base théorique de
toute cette comparaison.

## À retenir

Le protocole fixe données et hyperparamètres pour comparer SMO et LIBSVM. On peut reconstruire le dual depuis les coefficients des supports et examiner gap et KKT ; les temps mesurés restent propres à la machine et au protocole.

## Vérifiez votre compréhension

1. Pourquoi répéter les ajustements à froid ?

<details><summary>Réponse 1</summary>

Un seul ajustement très rapide est bruité ; répéter avec un nouvel estimateur rend le temps moyen plus lisible.

</details>

2. Comment reconstruire les multiplicateurs depuis `SVC` ?

<details><summary>Réponse 2</summary>

On place les valeurs absolues de `dual_coef_` aux indices `support_` et zéro ailleurs.

</details>

3. Une accuracy identique suffit-elle à établir une solution identique ?

<details><summary>Réponse 3</summary>

Non : comparer supports, scores de décision et gap apporte des contrôles complémentaires.

</details>

4. Que contrôle `tol` ?

<details><summary>Réponse 4</summary>

La tolérance d’arrêt du solveur : la resserrer demande en général plus de calcul pour une convergence plus précise.

</details>

## Pour aller plus loin

[SMO from scratch](2.7b-SMO-From-Scratch.ipynb) permet de relier les diagnostics à une implémentation explicite.
